# EWFS on a Quantum Computer

This notebook is a short demo of the main steps of the project. It follows the path used in the thesis: first the EWFS circuits are built, then they are simulated without noise, then they are transpiled for IBM Marrakesh, then a fake-hardware noise simulation is run, and finally the saved real-hardware paper data is loaded.

The notebook uses the project code in `src/ewfs`. Demo simulations are small and run in memory only. They do not create new run folders. The final hardware results are loaded from `data/paper_data`.

## Setup

First we load the project code and define the paths used below.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DEMO_SHOTS = 1024
PAPER_DATA = PROJECT_ROOT / "data" / "paper_data"

PROJECT_ROOT

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh

from ewfs.circuits.agents import AGENTS
from ewfs.experiments.ibm_transpilation import transpile_all_agents
from ewfs.analysis.agent_evaluation import (
    DATA_DIR_FAKE,
    DATA_DIR_NOISELESS,
    DATA_DIR_REAL,
    load_backend_result,
    load_backend_lf_series,
    result_display_label,
)

backend = FakeMarrakesh()
backend.name

## 1. Circuit Construction

The EWFS circuits are implemented in Qiskit. Barriers are used to keep the time ordering visible. Alice's and Bob's measurement choices are represented by two additional choice qubits. These choice qubits are put into a superposition with a Hadamard gate and then measured, giving random settings.

The setting choices are stored in the classical register `c`. Conditional blocks then choose which measurement setting is applied. Alice either reads the agent's memory or undoes the agent's measurement and measures the system directly. Bob's setting changes the measurement basis on `SB`.

Below we construct the four agent circuits directly from `agents.py` and display them in the notebook.

In [ ]:
circuit_rows = []
circuits = {}

for agent_name, build_circuit in AGENTS:
    circuit = build_circuit()
    circuits[agent_name] = circuit
    circuit_rows.append({
        "agent": agent_name,
        "qubits": circuit.num_qubits,
        "classical_bits": circuit.num_clbits,
        "depth": circuit.depth(),
        "operations": sum(circuit.count_ops().values()),
        "cx_count": circuit.count_ops().get("cx", 0),
    })

pd.DataFrame(circuit_rows)

In [ ]:
for agent_name, circuit in circuits.items():
    print(f"\n{agent_name}")
    print(circuit.draw(output="text", fold=120))

The Always 3/4 Agent and the Betting Agent are almost the same circuit. The important difference is the betting step. The Betting Agent uses the first memory result to decide the bet, while the Always 3/4 Agent always places the larger bet.

## 2. Noiseless Ideal Simulation

Before running on hardware, the circuits are tested with an ideal simulator. Here we use the Qiskit `AerSimulator` without a noise model. This means the circuit is simulated as if the gates were perfect. Measurements are still sampled, so the results depend on the number of shots.

In the thesis, the noiseless paper runs use many more shots. Here we run a small in-memory example so that the notebook stays quick and does not create new data folders.

In [ ]:
simulator = AerSimulator()
noiseless_data = {
    "kind": "demo_noiseless_simulation",
    "shots": DEMO_SHOTS,
    "agents": {},
}

for agent_name, build_circuit in AGENTS:
    circuit = build_circuit()
    counts = simulator.run(circuit, shots=DEMO_SHOTS).result().get_counts()
    noiseless_data["agents"][agent_name] = {
        "counts": {str(key): int(value) for key, value in counts.items()}
    }

print(f"Noiseless example with {noiseless_data['shots']} shots")
for agent_name, agent_data in noiseless_data["agents"].items():
    counts = agent_data["counts"]
    print(f"\n{agent_name}")
    print(f"  number of observed bitstrings: {len(counts)}")
    print(f"  total shots: {sum(counts.values())}")
    print(f"  five most common outcomes: {sorted(counts.items(), key=lambda item: item[1], reverse=True)[:5]}")

The paper-data noiseless runs are stored separately. These are the runs used for the thesis plots.

In [ ]:
paper_noiseless = load_backend_result(
    "Noiseless",
    DATA_DIR_NOISELESS,
    "noiseless_simulation.json",
)

print(result_display_label(paper_noiseless))
print(f"runs: {paper_noiseless['run_count']}")
print(f"total shots: {paper_noiseless['raw_shots_total']}")
print("run names:")
for run_name in paper_noiseless["run_names"]:
    print(f"  {run_name}")

## 3. Transpilation for IBM Marrakesh

IBM hardware only supports certain native gates and a fixed qubit connectivity. Therefore, the logical circuits have to be transpiled before they can run on the device.

For the thesis runs, the circuits are manually placed on physical qubits of `ibm_marrakesh`. The layout is chosen so that the needed two-qubit gates fit the hardware connectivity without adding SWAP gates. This matters because SWAP gates increase the circuit depth and therefore add more noise.

The transpilation below uses `optimization_level=0`, so Qiskit decomposes the gates into the device basis and respects the manual layout without trying to change the circuit logic for optimization. This is done in memory only.

In [ ]:
transpiled = transpile_all_agents(
    backend,
    save_plots=False,
)

transpile_rows = []
for agent_name, tqc in transpiled.items():
    ops = tqc.count_ops()
    transpile_rows.append({
        "agent": agent_name,
        "depth": tqc.depth(),
        "operations": sum(ops.values()),
        "cz_count": ops.get("cz", 0),
        "sx_count": ops.get("sx", 0),
        "rz_count": ops.get("rz", 0),
    })

pd.DataFrame(transpile_rows)

In [ ]:
for agent_name, circuit in transpiled.items():
    print(f"\n{agent_name}")
    print(circuit.draw(output="text", fold=120))

The next figure shows the Marrakesh qubits used by the different agent layouts. Green marks the system qubits, red marks the agent qubits, and yellow marks the random choice qubits.

In [ ]:
connectivity_plot = PROJECT_ROOT / "results" / "plots" / "plots_backend_connectivity" / "ibm_marrakesh_agent_connectivity.png"

if connectivity_plot.exists():
    display(Image(filename=str(connectivity_plot)))
else:
    print("Connectivity plot not found. Generate it with scripts/plot_ibm_connectivity.py if you want to display it here.")

## 4. Fake Hardware Simulation

The fake-hardware simulation uses a noise model built from the backend. This is still a simulation, but it includes device-like noise such as gate errors, relaxation effects, and readout errors. It gives an estimate of what may happen on hardware.

Here we run one small in-memory Marrakesh example using the transpiled circuits from above.

In [ ]:
noise_model = NoiseModel.from_backend(backend)
noise_simulator = AerSimulator(noise_model=noise_model)

fake_data = {
    "kind": "demo_fake_hardware_noise_sim",
    "backend": backend.name,
    "shots": DEMO_SHOTS,
    "agents": {},
}

for agent_name, transpiled_circuit in transpiled.items():
    counts = noise_simulator.run(transpiled_circuit, shots=DEMO_SHOTS).result().get_counts()
    fake_data["agents"][agent_name] = {
        "counts": {str(key): int(value) for key, value in counts.items()}
    }

print(f"Fake-hardware example on {fake_data['backend']} with {fake_data['shots']} shots")
for agent_name, agent_data in fake_data["agents"].items():
    counts = agent_data["counts"]
    print(f"\n{agent_name}")
    print(f"  number of observed bitstrings: {len(counts)}")
    print(f"  total shots: {sum(counts.values())}")
    print(f"  five most common outcomes: {sorted(counts.items(), key=lambda item: item[1], reverse=True)[:5]}")

The paper-data fake-hardware runs are the fixed Marrakesh noise-simulation runs used for the thesis plots.

In [ ]:
paper_fake = load_backend_result(
    "Fake hardware",
    DATA_DIR_FAKE,
    "fake_hardware_noise_sim.json",
)

print(result_display_label(paper_fake))
print(f"runs: {paper_fake['run_count']}")
print(f"total shots: {paper_fake['raw_shots_total']}")
print("run names:")
for run_name in paper_fake["run_names"]:
    print(f"  {run_name}")

## 5. Real Hardware Data

The real hardware jobs were run on IBM quantum hardware outside this notebook. This notebook does not submit new hardware jobs. Instead, it loads the archived Marrakesh paper runs from `data/paper_data/real_hardware`.

The hardware results include all physical noise present during the run. They can therefore differ from both the noiseless simulation and the fake-hardware noise simulation.

In [ ]:
paper_real = load_backend_result(
    "Real hardware",
    DATA_DIR_REAL,
    "real_hardware_run.json",
)

print(result_display_label(paper_real))
print(f"runs: {paper_real['run_count']}")
print(f"total shots: {paper_real['raw_shots_total']}")
print("run names:")
for run_name in paper_real["run_names"]:
    print(f"  {run_name}")

## 6. Compare the Saved Paper Results

Finally we collect the main values from the saved paper runs. The LF value `S` is calculated from the four correlators. Positive `S` means a violation in the convention used here.

In [ ]:
paper_results = [paper_noiseless, paper_fake, paper_real]

summary_rows = []
for result in paper_results:
    summary_rows.append({
        "data": result_display_label(result),
        "runs": result["run_count"],
        "shots": result["raw_shots_total"],
        "betting_payoff": result["observed_payoff"],
        "guessing_accuracy": result["guessing_accuracy"],
        "reflex_accuracy": result["reflex_accuracy"],
        "always_3_4_accuracy": result["always_large_accuracy"],
    })

pd.DataFrame(summary_rows)

In [ ]:
lf_rows = []
for result in paper_results:
    for agent_name, _ in AGENTS:
        series = load_backend_lf_series(paper_results, result["label"], agent_name)
        lf_rows.append({
            "data": result_display_label(result),
            "agent": agent_name,
            "S": series["_s_summary"]["value"],
            "S_stderr": series["_s_summary"]["stderr"],
            "E11": series["E11"]["value"],
            "E12": series["E12"]["value"],
            "E21": series["E21"]["value"],
            "E22": series["E22"]["value"],
        })

pd.DataFrame(lf_rows)